## 1. Configuración e Importaciones
En esta celda importamos las librerías y definimos las constantes del proyecto (nombres de datasets, algoritmos, etc.).

In [ ]:
import pandas as pd
import numpy as np
import joblib
import os
from scipy import stats
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score, confusion_matrix, roc_auc_score
import pathlib as pl

# CONFIGURACIÓN DE RUTAS ---
# AJUSTAR estas rutas según estructura de nuestras carpetas
TYPES = ['original', 'estandarizado', 'normalizado']
VARIANTS = ['', '_PCA95', '_PCA80']
FOLDER_PREFIX = 'conj'
VALIDATING_FILE_REGEX = 'validating*.csv'
DATA_PATH = './kfolds_data'
MODEL_PATH = './trained_models'
MODEL_EXT = 'joblib'                       
PATH_PREDICCIONES = "./predicciones/"                                   # Ruta donde se guardarán las predicciones  
PATH_METRICAS = "./metricas/"                                           # Ruta donde se guardarán las métricas                

# Crear carpetas si no existen
os.makedirs(PATH_PREDICCIONES, exist_ok=True)
os.makedirs(PATH_METRICAS, exist_ok=True)

# Lista con los algoritmos a evaluar y número de folds
MODELOS = ["KNN", "SVM", "NaiveBayes", "RandomForest"]

## 2. Funciones Auxiliares (Carga y Guardado)
Necesitamos funciones para cargar los datos y modelos correspondientes a cada iteración y almacenar los resultados.

In [ ]:
# Funciones de carga, evaluación y guardado de resultados
def cargar_datos_test(filename):
    try:
        df = pd.read_csv(filename)
        X = df.drop('species', axis=1)
        Y = df['species']
        return X, Y
    except FileNotFoundError:
        print(f"Falta archivo {filename}")
        return None, None

def cargar_modelo(filename):
    try:
        return joblib.load(filename)
    except FileNotFoundError:
        print(f"Falta modelo {filename}")
        return None

def calcular_metricas(y_true, y_pred, y_proba):
    """
    Calcula las métricas solicitadas.
    Adapta fórmulas binarias a multiclase usando macro-average.
    """
    # Métricas básicas de sklearn (usando 'macro' para multiclase)
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average='macro')      # F1-Score
    rec = recall_score(y_true, y_pred, average='macro') # Sensibilidad
    prec = precision_score(y_true, y_pred, average='macro') # Precisión
    
    # Métricas derivadas de la Matriz de Confusión (Especificidad, FNR, FPR)
    cm = confusion_matrix(y_true, y_pred)
    
    # Cálculo de TP, TN, FP, FN por clase
    FP = cm.sum(axis=0) - np.diag(cm)  
    FN = cm.sum(axis=1) - np.diag(cm)
    TP = np.diag(cm)
    TN = cm.sum() - (FP + FN + TP)

    # Evitar división por cero
    epsilon = 1e-7 
    
    # Promedios macro de las tasas
    spec_macro = np.mean(TN / (FP + TN + epsilon))
    fnr_macro = np.mean(FN / (TP + FN + epsilon))
    fpr_macro = np.mean(FP / (FP + TN + epsilon))
    
    # AUC 
    auc_val = 0
    if y_proba is not None:
        try:
            auc_val = roc_auc_score(y_true, y_proba, multi_class='ovr')
        except:
            pass

    return {
        "Exactitud": acc, "F1": f1, "Sensibilidad": rec, "Recall": rec,
        "Precision": prec, "Especificidad": spec_macro,
        "FNR": fnr_macro, "FPR": fpr_macro, "AUC": auc_val
    }

def guardar_resultados(dataset, fold, metodo, y_true, y_pred, y_proba):
    """
    Guarda DOS archivos: 
     Predicciones crudas (para ensembles y futuros análisis) y métricas calculadas (para la tabla de resultados finales)
    """
    # Guardar Predicciones (CSV Grande)
    data_pred = {"y_true": y_true, "y_pred": y_pred}
    if y_proba is not None:
        for i in range(y_proba.shape[1]):
            data_pred[f"prob_{i}"] = y_proba[:, i]
    
    pd.DataFrame(data_pred).to_csv(f"{PATH_PREDICCIONES}pred_{fold}_{dataset}_{metodo}.csv", index=False)

    # Guardar Métricas (CSV Pequeño con una fila) 
    metricas = calcular_metricas(y_true, y_pred, y_proba)
    metricas['Dataset'] = dataset
    metricas['Fold'] = fold
    metricas['Method'] = metodo
    
    # Reordenar para que las columnas identificadoras vayan primero
    cols = ['Dataset', 'Fold', 'Method'] + [k for k in metricas.keys() if k not in ['Dataset', 'Fold', 'Method']]
    
    pd.DataFrame([metricas], columns=cols).to_csv(f"{PATH_METRICAS}metrics_{fold}_{dataset}_{metodo}.csv", index=False)

## 3. Bucle Principal y Ensembles
Esta función implementa todas las fórmulas de la tabla del PDF. Al ser Iris un problema multiclase (3 clases), calculamos las métricas para cada clase y hacemos la media (Macro-average), que es el estándar cuando se piden estas fórmulas binarias en problemas multi-clase.

In [ ]:
for model in MODELOS:
    model_route = pl.Path(f'{MODEL_PATH}/{model}/')
    for tipo in TYPES:
            for var in VARIANTS:
                validating_route = pl.Path(f'{DATA_PATH}/{FOLDER_PREFIX}_{tipo}{var}/')
                model_folder = model_route.joinpath(f'{tipo}{var}')

                validating_csv = sorted([x.name for x in validating_route.glob(VALIDATING_FILE_REGEX)])
                validating_models = sorted([x.name for x in model_folder.glob(f'{model}*.{MODEL_EXT}')])

In [ ]:
print("--- INICIANDO EVALUACIÓN Y CÁLCULO DE MÉTRICAS ---")

for dataset in DATASETS:
    for fold in FOLDS:
        X_test, y_test = cargar_datos_test(dataset, fold)
        if X_test is None: continue
        
        ensemble_clases = []
        ensemble_probas = []
        modelos_ok = 0
        
        # Modelos Individuales
        for algo in ALGORITMOS:
            model = cargar_modelo(dataset, algo, fold)
            if model:
                try:
                    y_pred = model.predict(X_test)
                    
                    # Probabilidades (Vital para Ensembles Media/Mediana y AUC)
                    if hasattr(model, "predict_proba"):
                        y_proba = model.predict_proba(X_test)
                    else:
                        y_proba = np.zeros((len(y_pred), len(np.unique(y_test))))
                    
                    guardar_resultados(dataset, fold, algo, y_test, y_pred, y_proba)
                    
                    ensemble_clases.append(y_pred)
                    ensemble_probas.append(y_proba)
                    modelos_ok += 1
                except Exception as e:
                    print(f"Error {algo} {dataset} fold {fold}: {e}")

        # Ensembles (Solo si tenemos los 4 modelos base)
        if modelos_ok == 4:
            # 1. Votación 
            moda = stats.mode(np.array(ensemble_clases), axis=0, keepdims=True)
            y_votacion = moda[0][0]
            y_proba_votacion = np.mean(np.array(ensemble_probas), axis=0) # Dummy proba para formato
            guardar_resultados(dataset, fold, "Ensemble_Votacion", y_test, y_votacion, y_proba_votacion)
            
            # 2. Media 
            y_proba_media = np.mean(np.array(ensemble_probas), axis=0)
            y_pred_media = np.argmax(y_proba_media, axis=1)
            guardar_resultados(dataset, fold, "Ensemble_Media", y_test, y_pred_media, y_proba_media)
            
            # 3. Mediana 
            y_proba_mediana = np.median(np.array(ensemble_probas), axis=0)
            y_pred_mediana = np.argmax(y_proba_mediana, axis=1)
            guardar_resultados(dataset, fold, "Ensemble_Mediana", y_test, y_pred_mediana, y_proba_mediana)

print("--- EVALUACIÓN COMPLETADA ---")